# 文本預處理
:label:`sec_text_preprocessing`

對於序列數據處理問題，我們在 :numref:`sec_sequence`中
評估了所需的統計工具和預測時面臨的挑戰。
這樣的數據存在許多種形式，文本是最常見例子之一。
例如，一篇文章可以被簡單地看作一串單詞序列，甚至是一串字符序列。
本節中，我們將解析文本的常見預處理步驟。
這些步驟通常包括：

1. 將文本作為字串加載到記憶體中。
1. 將字串拆分為詞元（如單詞和字符）。
1. 建立一個詞表，將拆分的詞元映射到數字索引。
1. 將文本轉換為數字索引序列，方便模型操作。


In [1]:
import collections
import re

## 讀取數據集

首先，我們從H.G.Well的[時光機器](https://www.gutenberg.org/ebooks/35)中加載文本。
這是一個相當小的語料庫，只有30000多個單詞，但足夠我們小試牛刀，
而現實中的文件集合可能會包含數十億個單詞。
下面的函數(**將數據集讀取到由多條文本行組成的列表中**)，其中每條文本行都是一個字串。
為簡單起見，我們在這裡忽略了標點符號和字母大寫。


In [3]:
import re
import requests

def read_time_machine(url="http://www.gutenberg.org/files/35/35-0.txt"):
    """將時光機器數據集加載到文本行的列表中"""
    response = requests.get(url)
    response.encoding = 'utf-8'  # 設定編碼為 UTF-8
    lines = response.text.splitlines()
    return [re.sub(r'[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

lines = read_time_machine()
print(f'# 文本總行數: {len(lines)}')
print(lines[0])
print(lines[10])

# 文本總行數: 3557
the project gutenberg ebook of the time machine by h g wells
title the time machine


## 詞元化

下面的`tokenize`函數將文本行列表（`lines`）作為輸入，
列表中的每個元素是一個文本序列（如一條文本行）。
[**每個文本序列又被拆分成一個詞元列表**]，*詞元*（token）是文本的基本單位。
最後，返回一個由詞元列表組成的列表，其中的每個詞元都是一個字串（string）。


In [4]:
def tokenize(lines, token='word'):  #@save
    """將文本行拆分為單詞或字符詞元"""
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('錯誤：未知詞元類型：' + token)

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
['this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'in', 'the', 'united', 'states', 'and']
['most', 'other', 'parts', 'of', 'the', 'world', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions']
['whatsoever', 'you', 'may', 'copy', 'it', 'give', 'it', 'away', 'or', 're', 'use', 'it', 'under', 'the', 'terms']
['of', 'the', 'project', 'gutenberg', 'license', 'included', 'with', 'this', 'ebook', 'or', 'online', 'at']
['www', 'gutenberg', 'org', 'if', 'you', 'are', 'not', 'located', 'in', 'the', 'united', 'states', 'you']
['will', 'have', 'to', 'check', 'the', 'laws', 'of', 'the', 'country', 'where', 'you', 'are', 'located', 'before']
['using', 'this', 'ebook']
[]
['title', 'the', 'time', 'machine']


## 詞表

詞元的類型是字串，而模型需要的輸入是數字，因此這種類型不方便模型使用。
現在，讓我們[**構建一個字典，通常也叫做*詞表*（vocabulary），
用來將字串類型的詞元映射到從$0$開始的數字索引中**]。
我們先將訓練集中的所有文件合併在一起，對它們的唯一詞元進行統計，
得到的統計結果稱之為*語料*（corpus）。
然後根據每個唯一詞元的出現頻率，為其分配一個數字索引。
很少出現的詞元通常被移除，這可以降低複雜性。
另外，語料庫中不存在或已刪除的任何詞元都將映射到一個特定的未知詞元“&lt;unk&gt;”。
我們可以選擇增加一個列表，用於保存那些被保留的詞元，
例如：填充詞元（“&lt;pad&gt;”）；
序列開始詞元（“&lt;bos&gt;”）；
序列結束詞元（“&lt;eos&gt;”）。


In [5]:
class Vocab:  #@save
    """文本詞表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        # 按出現頻率排序
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                   reverse=True)
        # 未知詞元的索引為0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx
                             for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):  # 未知詞元的索引為0
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

def count_corpus(tokens):  #@save
    """統計詞元的頻率"""
    # 這裡的tokens是1D列表或2D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 將詞元列表展平成一個列表
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)

我們首先使用時光機器數據集作為語料庫來[**構建詞表**]，然後打印前幾個高頻詞元及其索引。


In [6]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

[('<unk>', 0), ('the', 1), ('and', 2), ('of', 3), ('i', 4), ('a', 5), ('to', 6), ('in', 7), ('was', 8), ('that', 9)]


現在，我們可以(**將每條文本行轉換成一個數字索引列表**)。


In [7]:
for i in [0, 10]:
    print('文本:', tokens[i])
    print('索引:', vocab[tokens[i]])

文本: ['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'time', 'machine', 'by', 'h', 'g', 'wells']
索引: [1, 53, 44, 314, 3, 1, 19, 46, 33, 1163, 1164, 360]
文本: ['title', 'the', 'time', 'machine']
索引: [2445, 1, 19, 46]


## 整合所有功能

在使用上述函數時，我們[**將所有功能打包到`load_corpus_time_machine`函數中**]，
該函數返回`corpus`（詞元索引列表）和`vocab`（時光機器語料庫的詞表）。
我們在這裡所做的改變是：

1. 為了簡化後面章節中的訓練，我們使用字符（而不是單詞）實現文本詞元化；
1. 時光機器數據集中的每條文本行不一定是一個句子或一個段落，還可能是一個單詞，因此返回的`corpus`僅處理為單個列表，而不是使用多詞元列表構成的一個列表。


In [8]:
def load_corpus_time_machine(max_tokens=-1):  #@save
    """返回時光機器數據集的詞元索引列表和詞表"""
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    # 因為時光機器數據集中的每條文本行不一定是一個句子或一個段落，
    # 所以將所有文本行展平到一個列表中
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)

(189663, 28)

## 小結

* 文本是序列數據的一種最常見的形式之一。
* 為了對文本進行預處理，我們通常將文本拆分為詞元，構建詞表將詞元字符串映射為數字索引，並將文本數據轉換為詞元索引以供模型操作。

## 練習

1. 詞元化是一個關鍵的預處理步驟，它因語言而異。嘗試找到另外三種常用的詞元化文本的方法。
1. 在本節的實驗中，將文本詞元為單詞和更改`Vocab`實例的`min_freq`參數。這對詞表大小有何影響？


[Discussions](https://discuss.d2l.ai/t/2094)
